# Introduction - SABR Calibration and Validation
This notebook calibrates SABR to the SSVI surface and validates the fit.

# Import Modules

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.connector_factory import ConnectorFactory
from src.models.bsm import implied_volatility
from src.models.svi import fit_svi_smile, svi_total_variance
from src.models.ssvi import calibrate_ssvi, ssvi_iv
from src.models.sabr import calibrate_sabr, sabr_iv_vectorized
from src.utils.time_utils import compute_time_to_expiry
from src.utils.data_preprocessing import filter_otm_for_calibration

np.random.seed(42)


# Configure the notebook and setup variables

In [2]:
SYMBOL = "SPY"
MAX_EXPIRIES = 5
R = 0.045
Q = 0.012

connector = ConnectorFactory.get_connector(provider="yfinance", symbol=SYMBOL)
spot = connector.get_spot_price()
forward = spot*np.exp(R-Q)

print(f"Spot: {spot:.2f}    -   Forward: {forward:.2f}")

Spot: 747.71    -   Forward: 772.80


# Fetch data and calibrate SSVI (reuse logic from SSVI notebook)

In [3]:
expiries = connector.get_available_expiries()[1:MAX_EXPIRIES+1]
raw_chains = {exp: connector.get_chain_for_expiry(exp) for exp in expiries}

def prepare_svi_slice(expiry, raw_data):
    T = compute_time_to_expiry(expiry)
    forward = spot * np.exp((R - Q) * T)
    all_ops = raw_data["calls"] + raw_data["puts"]
    for opt in all_ops:
        if opt.mid > 0 and opt.bid > 0 and opt.ask > 0:
            iv = implied_volatility(opt.mid, spot, opt.strike, T, R, Q, opt.option_type)
            if not np.isnan(iv):
                opt.implied_vol = iv
    strikes, ivs = filter_otm_for_calibration(all_ops, spot, T, R, Q, max_spread_pct=0.5)
    if len(strikes) < 5:
        return None
    result = fit_svi_smile(strikes, ivs, T, forward, r=R, q=Q)
    if not result.success:
        return None
    a, b, rho, m, sigma = result.params
    theta = svi_total_variance(0, a, b, rho, m, sigma)
    k = np.log(strikes / forward)
    return {
        "expiry": expiry,
        "T": T,
        "forward": forward,
        "strikes": strikes,
        "ivs": ivs,
        "k": k,
        "theta": theta,
    }

svi_slices = []
for exp, raw in raw_chains.items():
    slice_data = prepare_svi_slice(exp, raw)
    if slice_data is not None:
        svi_slices.append(slice_data)

print(f"Prepared {len(svi_slices)} SVI slices.")

ssvi_input = {}
for s in svi_slices:
    ssvi_input[s["expiry"]] = {
        "k": s["k"],
        "iv": s["ivs"],
        "theta": s["theta"],
        "T": s["T"],
    }

gamma = 0.5
ssvi_result = calibrate_ssvi(ssvi_input, gamma=gamma)
rho_ssvi, eta_ssvi = ssvi_result.params

if ssvi_result.success:
    rho, eta = ssvi_result.params
    print(f"\n✅ SSVI Calibration SUCCESSFUL!")
    print(f"   rho (correlation): {rho_ssvi:.4f}")
    print(f"   eta (skew decay): {eta_ssvi:.4f}")
    print(f"   gamma (fixed): {gamma:.1f}")
    print(f"   Message: {ssvi_result.message}")
else:
    print(f"\n❌ SSVI Calibration FAILED: {ssvi_result.message}")
    raise RuntimeError("Stop here.")

Failed to fetch chain for 2026-07-14: cannot convert float NaN to integer
Failed to fetch chain for 2026-07-15: cannot convert float NaN to integer
Prepared 3 SVI slices.

✅ SSVI Calibration SUCCESSFUL!
   rho (correlation): -0.5854
   eta (skew decay): 1.1049
   gamma (fixed): 0.5
   Message: Calibration successful. Iterations: 8


# Calibrate SABR to each SSVI slice

In [4]:
sabr_results = {}
for s in svi_slices:
    # Extract parameters from SVI slice
    strikes = s["strikes"]
    ivs = s["ivs"]
    T = s["T"]

    # Use SSVI fitted IVs as target (smooth surface)
    k = s["k"]
    iv_ssvi_target = ssvi_iv(k, s["theta"], rho_ssvi, eta_ssvi, T, gamma)
    
    # Calibrate SABR to the SSVI IVs
    result = calibrate_sabr(strikes, iv_ssvi_target, T, forward=forward, beta=0.5)
    sabr_results[s["expiry"]] = {
        "result": result,
        "strikes": strikes,
        "iv_market": ivs,
        "iv_ssvi": iv_ssvi_target,
    }
    if result.success:
        alpha, rho, nu = result.params
        print(f"{s['expiry']}: alpha={alpha:.4f}, rho={rho:.4f}, nu={nu:.4f}")
    else:
        print(f"{s['expiry']}: FAILED")

2026-07-09: alpha=1.4981, rho=0.3581, nu=17.7623
2026-07-10: alpha=3.3580, rho=0.4823, nu=11.9836
2026-07-13: alpha=2.6014, rho=0.5397, nu=8.3667


# Visualization: Market IV vs SSVI vs SABR

In [8]:
num_slices = len(svi_slices)
fig = make_subplots(rows=1, cols=num_slices, shared_yaxes=True, horizontal_spacing=0.08,
                    subplot_titles=[f"{s['expiry']} (T={s['T']*365:.3f} days)" for s in svi_slices])

for i, s in enumerate(svi_slices):
    strikes = s["strikes"]
    iv_market = s["ivs"]
    T = s["T"]
    k = s["k"]
    iv_ssvi = ssvi_iv(k, s["theta"], rho_ssvi, eta_ssvi, T, gamma)
    
    sabr_res = sabr_results[s["expiry"]]["result"]
    if sabr_res.success:
        alpha, rho, nu = sabr_res.params
        f = strikes[np.argmin(iv_market)]  # Approx forward
        iv_sabr = sabr_iv_vectorized(f, strikes, T, alpha, 0.5, rho, nu)
    else:
        iv_sabr = np.full_like(strikes, np.nan)
    
    fig.add_trace(go.Scatter(x=strikes, y=iv_market, mode='markers', name='Market IV', showlegend=(i == 0), marker=dict(color='blue', size=6)), row=1, col=i+1)
    fig.add_trace(go.Scatter(x=strikes, y=iv_ssvi, mode='markers', name='SSVI', showlegend=(i == 0), line=dict(color='red', width=2)), row=1, col=i+1)
    fig.add_trace(go.Scatter(x=strikes, y=iv_sabr, mode='markers', name='SABR', showlegend=(i == 0), line=dict(color='green', width=2, dash='dot')), row=1, col=i+1)

fig.update_layout(title="SABR Fit vs SSVI vs Market IVs", height=500, width=1200)
fig.show()

# Dynamic Shifting Demo

In [6]:
expiry = svi_slices[0]["expiry"]
sabr_res = sabr_results[expiry]["result"]
if sabr_res.success:
    alpha, rho, nu = sabr_res.params
    f_base = svi_slices[0]["forward"]
    strikes = svi_slices[0]["strikes"]
    
    # Shift spot by +1%
    f_new = f_base * 1.01
    iv_shifted = sabr_iv_vectorized(f_new, strikes, svi_slices[0]["T"], alpha, 0.5, rho, nu)
    iv_orig_sabr = sabr_iv_vectorized(f_base, strikes, svi_slices[0]["T"], alpha, 0.5, rho, nu)
    
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=strikes, y=iv_shifted, mode='markers', name=f'Spot +1% (F={f_new:.2f})'))
    fig2.add_trace(go.Scatter(x=strikes, y=iv_orig_sabr, mode='markers', name='Base F', line=dict(dash='dot')))
    fig2.update_layout(title=f"SABR Dynamic Re-pricing: Spot Shift +1% for {expiry}")
    fig2.show()
else:
    print(f"ERROR: SABR was not successfull -   Error Messag: {sabr_res.message}")